# Nestling Plotting Tutorial

A companion notebook to the [Nestling plotting track](https://meighenbergs.github.io/nestling/tracks/06-plotting/).  
It walks through all four lessons with live, rendered output.

**Acknowledgement:** Several examples and design choices draw on Ciaran O'Hare's  
[HowToMakeAPlot](https://github.com/cajohare/HowToMakeAPlot).

---

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

STYLE = Path("..") / "styles" / "nestling.mplstyle"
plt.style.use(str(STYLE))

# Physics parameters used throughout
PHI0  = 1.44e-18   # normalisation at E0 = 100 TeV [GeV^{-1} cm^{-2} s^{-1} sr^{-1}]
E0    = 1e5        # 100 TeV in GeV
GAMMA = 2.37       # IceCube best-fit spectral index (approximate, for illustration)

def power_law(E, gamma=GAMMA):
    return PHI0 * (E / E0) ** (-gamma)

DARK2 = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02"]

rng = np.random.default_rng(0)

---

## Lesson 01 — Figure Design Principles

The two cells below show the same neutrino spectrum **before** and **after** applying
the principles from Lesson 01: remove chartjunk, add units, use inward mirror ticks.

In [ ]:
E = np.logspace(3, 7, 300)

# BEFORE: matplotlib defaults with common mistakes
with plt.rc_context({
    "font.family": "sans-serif", "axes.grid": True,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.top": False, "ytick.right": False,
    "legend.frameon": True, "font.size": 12,
}):
    fig, ax = plt.subplots()
    ax.loglog(E, E**2 * power_law(E))
    ax.set_title("Neutrino Flux vs Energy")   # title inside panel — avoid
    ax.set_xlabel("Energy")                   # no units
    ax.set_ylabel("Flux")                     # no units, no symbol
    ax.legend(["spectrum"])
    fig.suptitle("BEFORE — common mistakes", y=1.02, fontsize=9, color="firebrick")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# AFTER: Nestling style, correct labels, no chartjunk
fig, ax = plt.subplots()
ax.loglog(E, E**2 * power_law(E), label=r"$E^{-2.37}$ spectrum")
ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax.legend()
fig.suptitle("AFTER — Nestling style", y=1.02, fontsize=9, color="#1b9e77")
plt.tight_layout()
plt.show()
plt.close(fig)

---

## Lesson 02 — matplotlib Basics

### Line plot

In [ ]:
gammas = [2.0, 2.37, 2.7]

fig, ax = plt.subplots()
for gamma in gammas:
    ax.loglog(E, E**2 * power_law(E, gamma), label=rf"$\gamma = {gamma}$")
ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
x = np.linspace(0, 2 * np.pi, 200)

fig, ax = plt.subplots()
ax.plot(x, np.sin(x))
ax.plot(x, np.cos(x))

# Direct annotation — preferred over a legend for a small number of curves
ax.text(2.9, np.sin(3), r"$\sin(x)$", va="center", fontsize=7, rotation=-70, color=DARK2[3])
ax.text(2.7, np.cos(3) + 0.16, r"$\cos(x)$", va="center", fontsize=7, color=DARK2[0])

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_xlim(0, x[-1])
plt.tight_layout()
plt.show()
plt.close(fig)

### Scatter plot with colorbar

### Labelling curves: annotations over legends

A direct annotation close to the curve is almost always cleaner than a legend box —
no eye-travel back and forth, and no whitespace reserved for the legend.
The Nestling style already removes the legend frame, but consider going further.

In [ ]:
n = 300
zenith  = rng.uniform(0, 180, n)
azimuth = rng.uniform(0, 360, n)
log_E   = rng.uniform(2, 6, n)

fig, ax = plt.subplots()
sc = ax.scatter(azimuth, np.cos(np.radians(zenith)),
                c=log_E, cmap="viridis", s=8, linewidths=0)
cb = plt.colorbar(sc, ax=ax)
cb.set_label(r"$\log_{10}(E\,/\,\mathrm{GeV})$")
ax.set_xlabel(r"azimuth / ${}^{\circ}$")
ax.set_ylabel(r"$\cos\,\theta_\mathrm{zen}$")
ax.set_xlim(0, 360)
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.show()
plt.close(fig)

### Overlapping histograms

In [ ]:
muons     = rng.exponential(scale=3.5, size=4000)
electrons = rng.exponential(scale=1.5, size=4000)
bins = np.linspace(0, 10, 40)

fig, ax = plt.subplots()
ax.hist(muons,     bins=bins, histtype="step", density=True, label="muons")
ax.hist(electrons, bins=bins, histtype="step", density=True,
        linestyle="--", label="electrons")
ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"probability density / GeV$^{-1}$")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

### Uncertainty band

In [ ]:
central = E**2 * power_law(E, GAMMA)
lower   = E**2 * power_law(E, GAMMA + 0.13)
upper   = E**2 * power_law(E, GAMMA - 0.13)

fig, ax = plt.subplots()
ax.fill_between(E, lower, upper, alpha=0.25, label=r"$1\sigma$ uncertainty")
ax.loglog(E, central, label=r"best fit $\gamma = 2.37$")
ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

### Multi-panel figure with residuals

In [ ]:
E_bins = np.logspace(3, 7, 16)
E_mid  = np.sqrt(E_bins[:-1] * E_bins[1:])

# Normalise so the brightest bin has ~50 counts, then Poisson-fluctuate
model_at_mid = power_law(E_mid) * E_mid**2
scale = 50.0 / model_at_mid.max()
lam  = model_at_mid * scale
data = rng.poisson(lam) / scale
err  = np.sqrt(lam) / scale

E_fine = np.logspace(3, 7, 400)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(3.46, 4.0), sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
)

ax_top.loglog(E_fine, power_law(E_fine) * E_fine**2, label="best fit")
ax_top.errorbar(E_mid, data, yerr=err, fmt="o", markersize=3,
                linewidth=0.8, label="simulated data")
ax_top.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax_top.legend()

residuals = (data - model_at_mid) / model_at_mid
res_err   = err / model_at_mid
ax_bot.axhline(0, linewidth=0.8, linestyle="--")
ax_bot.errorbar(E_mid, residuals, yerr=res_err, fmt="o", markersize=3, linewidth=0.8)
ax_bot.set_xlabel(r"$E$ / GeV")
ax_bot.set_ylabel("residual")
ax_bot.set_ylim(-0.8, 0.8)

plt.tight_layout()
plt.show()
plt.close(fig)

---

## Lesson 03 — Publication Style

The Nestling style is already active (`plt.style.use(STYLE)` in the setup cell).  
Here we demonstrate LaTeX labels and figure sizing for one- and two-column layouts.

In [ ]:
# LaTeX labels using matplotlib's built-in math renderer (no local TeX required)
fig, ax = plt.subplots()
ax.loglog(E, E**2 * power_law(E))
ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax.set_title(r"text.usetex = False  (matplotlib math renderer)", fontsize=7)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Two-column layout (figsize matches a full journal page width)
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8))

for ax, gamma, label in zip(
    axes,
    [2.0, 2.7],
    [r"hard spectrum $\gamma=2.0$", r"soft spectrum $\gamma=2.7$"],
):
    ax.loglog(E, E**2 * power_law(E, gamma))
    ax.set_xlabel(r"$E$ / GeV")
    ax.set_title(label, fontsize=7)

axes[0].set_ylabel(r"$E^2\,\phi_\nu$")
plt.tight_layout()
plt.show()
plt.close(fig)

---

## Lesson 04 — Colour and Accessibility

### Dark2 (CVD-safe) vs jet-sampled colours

In [ ]:
_jet = plt.get_cmap("jet")
JET6 = [_jet(v) for v in np.linspace(0.05, 0.95, 6)]
gammas6 = np.linspace(2.0, 2.7, 6)

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(6.4, 3.0), sharey=True)

for i, gamma in enumerate(gammas6):
    label = rf"$\gamma={gamma:.1f}$"
    ax_l.loglog(E, E**2 * power_law(E, gamma), color=DARK2[i], label=label)
    ax_r.loglog(E, E**2 * power_law(E, gamma), color=JET6[i],  label=label)

for ax, title in zip([ax_l, ax_r], ["Dark2 (CVD-safe)", "jet-sampled"]):
    ax.set_xlabel(r"$E$ / GeV")
    ax.set_title(title, fontsize=8)
    ax.legend(fontsize=6, handlelength=1.5)

ax_l.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
plt.tight_layout()
plt.show()
plt.close(fig)

### Sequential colourmap: viridis vs jet

In [ ]:
x = rng.standard_normal(30_000)
y = rng.standard_normal(30_000) + 0.5 * x

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(6.4, 2.8))

for ax, cmap, title in zip(
    [ax_l, ax_r],
    ["viridis", "jet"],
    ["viridis (perceptually uniform)", "jet (avoid)"],
):
    _, _, _, img = ax.hist2d(x, y, bins=50, cmap=cmap)
    plt.colorbar(img, ax=ax, label="counts")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_title(title, fontsize=8)

plt.tight_layout()
plt.show()
plt.close(fig)

### Redundant encoding — colour + line style

In [ ]:
linestyles = ["-", "--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 2))]

fig, ax = plt.subplots()
for i, (gamma, ls) in enumerate(zip(gammas6, linestyles)):
    ax.loglog(E, E**2 * power_law(E, gamma),
              color=DARK2[i], linestyle=ls,
              label=rf"$\gamma={gamma:.1f}$")

ax.set_xlabel(r"$E$ / GeV")
ax.set_ylabel(r"$E^2\,\phi_\nu$ / GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$")
ax.legend(fontsize=6, handlelength=2.5)
plt.tight_layout()
plt.show()
plt.close(fig)

---

## Resources

- [ColorBrewer 2.0](https://colorbrewer2.org/#type=qualitative&scheme=Set1&n=7) — palette explorer, filtered by CVD-safe and print-safe.
- [Ciaran O'Hare — HowToMakeAPlot](https://github.com/cajohare/HowToMakeAPlot) — worked matplotlib examples.
- [matplotlib colormap reference](https://matplotlib.org/stable/gallery/color/colormap_reference.html)
- [Scientific colour maps (Crameri)](https://www.fabiocrameri.ch/colourmaps/) — perceptually uniform, CVD-safe.
- [Coblis CVD simulator](https://www.color-blindness.com/coblis-color-blindness-simulator/) — upload any figure for CVD preview.